# 07 — Falsification & Validation

Tests that could kill the theory:
- **Test A**: Random taxonomy baseline — do 9 random categories produce similar z-scores?
- **Test B**: Missing operator detection — do homeless verbs cluster together?
- **Test C**: Is k=9 special? — merge/split operators and compare z-scores
- **Test D**: Defection analysis — which verbs are geometrically closer to another operator?

**Requires:** Embeddings and classifications (pre-computed or generated).

In [ ]:
# ═══ Load falsification data ═══
import json
from pyodide.http import pyfetch

report = None
try:
    resp = await pyfetch('../output/falsification_report.json')
    report = json.loads(await resp.string())
    print('Loaded pre-computed falsification report')
except:
    print('No pre-computed falsification report.')
    print('Running live falsification requires embeddings + classifications.')

In [ ]:
# ═══ TEST A: Random Taxonomy Baseline ═══
if report and 'test_a_random_baseline' in report:
    ta = report['test_a_random_baseline']
    print('TEST A: RANDOM TAXONOMY BASELINE')
    print('=' * 50)
    print(f'If 9 random categories produce similar z-scores to EO,')
    print(f'the pipeline finds structure in anything and EO is noise.')
    print()
    print(f'  EO z-score:           {ta["eo_z_score"]:>+8.1f}')
    print(f'  Random taxonomy mean: {ta["random_taxonomy_z_mean"]:>+8.1f}')
    print(f'  Random taxonomy std:  {ta["random_taxonomy_z_std"]:>8.1f}')
    print(f'  Random range:         [{ta["random_taxonomy_z_range"][0]:+.1f}, {ta["random_taxonomy_z_range"][1]:+.1f}]')
    print(f'  EO vs Random ratio:   {ta["eo_vs_random_ratio"]:>8.0f}x')
    print()
    if ta['eo_vs_random_ratio'] > 5:
        print('  RESULT: EO captures structure that random categories do NOT')
    else:
        print('  WARNING: Random categories show similar structure')
else:
    print('Test A data not available.')

In [ ]:
# ═══ TEST B/D: Defection Analysis ═══
if report and 'test_b_defectors' in report:
    tb = report['test_b_defectors']
    print('TEST D: DEFECTION ANALYSIS')
    print('=' * 50)
    print(f'Verbs geometrically closer to a DIFFERENT operator than assigned.')
    print()
    print(f'  Total defectors: {tb["n_defectors"]} / {tb["n_total"]} ({tb["defection_rate"]*100:.1f}%)')
    print()
    if tb['defection_rate'] < 0.15:
        print('  RESULT: Low defection rate — operator assignments are geometrically consistent')
    elif tb['defection_rate'] < 0.30:
        print('  RESULT: Moderate defection — some boundary verbs are compositional')
    else:
        print('  WARNING: High defection rate — operator boundaries may need refinement')
else:
    print('Defection data not available.')

In [ ]:
# ═══ Run live falsification (if embeddings available) ═══
import numpy as np

try:
    verb_embeddings
    all_classifications
    
    HELIX = ['NUL','DES','INS','SEG','CON','SYN','ALT','SUP','REC']
    labels = np.array([HELIX.index(c['operator']) if c.get('operator') in HELIX else -1 
                       for c in all_classifications])
    
    # Ensure same length
    n = min(len(verb_embeddings), len(labels))
    embs = verb_embeddings[:n]
    labs = labels[:n]
    
    print(f'Running falsification on {n} verbs...')
    
    from analysis import test_falsification
    result = test_falsification(embs, labs, n_random_taxonomies=10)
    
    print(f'\nResults:')
    print(f'  EO z-score: {result["eo_z_score"]:.1f}')
    print(f'  Random mean: {result["random_z_mean"]:.1f}')
    print(f'  EO vs random: {result["eo_vs_random_ratio"]:.0f}x')
    print(f'  Verdict: {result["verdict"]}')

except NameError:
    print('Need verb_embeddings and all_classifications to run live tests.')
    print('Run notebooks 02 and 04 first, or use the pre-computed results above.')
except Exception as e:
    print(f'Error: {e}')